# 04 — Políticas e presets: configurar o **comportamento** dos modelos

Os notebooks anteriores carregavam **dados** de configuração. Aqui a configuração descreve **como os modelos se comportam**:
o `model_config` do Pydantic e as opções do `@dataclass`. É a ideia que começou com o arquivo [`configs/data_model.yaml`](configs/data_model.yaml).

```
presets em código (classe base / partial)  ──  o padrão; revisável, tipado
        │ quando há motivo real para sair do código
        ▼
arquivo de perfis ──► ModelPolicy / DataclassPolicy (validação do PRÓPRIO arquivo) ──► ConfigDict / opções do @dataclass
```

In [1]:
from dataclasses import dataclass, fields
from enum import Enum, StrEnum
from functools import partial
from pathlib import Path
from typing import Any, Literal, Self

import yaml
from pydantic import BaseModel, ConfigDict, ValidationError, model_validator
from pydantic.alias_generators import to_camel, to_pascal, to_snake

CONFIGS = Path("configs")

## 1. O padrão: presets em código

Uma classe base por perfil (Pydantic) e um `partial` por perfil (dataclass). Sem I/O, sem arquivo, com autocompletar e revisão.

In [2]:
class ApiModel(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True, validate_assignment=True,
                              alias_generator=to_camel, validate_by_name=True)


class InternalModel(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True, strict=True)


value_object = partial(dataclass, frozen=True, slots=True, kw_only=True)


class Order(ApiModel):
    customer_name: str
    total_cents: int


@value_object
class Money:
    amount_cents: int
    currency: str


print(f"Order.model_validate({{'customerName': ' Ada ', 'totalCents': '1500'}}): {Order.model_validate({'customerName': ' Ada ', 'totalCents': '1500'})!r}")
print(f"Money(amount_cents=100, currency='BRL'): {Money(amount_cents=100, currency='BRL')!r}")
try:
    Money(100, "BRL")
except TypeError as ex:
    print(f"kw_only: {ex}")

Order.model_validate({'customerName': ' Ada ', 'totalCents': '1500'}): Order(customer_name='Ada', total_cents=1500)
Money(amount_cents=100, currency='BRL'): Money(amount_cents=100, currency='BRL')
kw_only: Money.__init__() takes 1 positional argument but 3 were given


Na grande maioria dos projetos, **é aqui que a história termina**. O resto do notebook é para quando há motivo real para tirar a
política do código (muitos serviços compartilhando perfis, modelos gerados — notebook [05](05_modelos_a_partir_de_especificacao.ipynb)).

## 2. O arquivo original: `data_model.yaml` aplicado direto

In [3]:
raw_policy = yaml.safe_load((CONFIGS / "data_model.yaml").read_text())["data_config"]
print(f"raw_policy: {raw_policy}")


class Color(Enum):
    RED = "red"


class Direct(BaseModel):
    model_config = ConfigDict(**raw_policy)
    code: str
    color: Color


for label, attempt in [
    ("dict, code=10", lambda: Direct.model_validate({"code": 10, "color": Color.RED})),
    ("dict, color='red'", lambda: Direct.model_validate({"code": "10", "color": "red"})),
    ("json, color='red'", lambda: Direct.model_validate_json('{"code": "10", "color": "red"}')),
]:
    try:
        print(f"{label}: {attempt()!r}")
    except ValidationError as ex:
        print(f"{label}: erro {[e['type'] for e in ex.errors()]}")

raw_policy: {'validate_assignment': True, 'validate_default': True, 'extra': 'allow', 'frozen': False, 'strict': True, 'coerce_numbers_to_str': True, 'str_strip_whitespace': True, 'str_to_lower': True, 'str_to_upper': False, 'use_enum_values': True, 'populate_by_name': True}
dict, code=10: erro ['string_type']
dict, color='red': erro ['is_instance_of']
json, color='red': Direct(code='10', color='red')


Três achados, nenhum deles com aviso:

| achado | efeito |
|---|---|
| `strict: True` + `coerce_numbers_to_str: True` | `strict` vence; `code=10` é rejeitado — a segunda opção é letra morta |
| `strict: True` + enum | `"red"` é rejeitado num `dict`, mas aceito vindo de JSON: o mesmo modelo muda de comportamento com a origem |
| `populate_by_name` | funciona, mas o nome atual é `validate_by_name` (2.11+) |

E o problema estrutural: um erro de digitação passa **em silêncio**:

In [4]:
class Typo(BaseModel):
    model_config = ConfigDict(**{"strct": True})
    x: int


print(f"Typo(x='1') com 'strct': {Typo(x='1')!r}")

Typo(x='1') com 'strct': Typo(x=1)


## 3. Validando o próprio arquivo: `ModelPolicy`

`TypeAdapter(ConfigDict)` não consegue validar o arquivo (a construção do adaptador falha). A saída é um modelo nosso que descreve
**o subconjunto permitido** — chaves fechadas, tipos estritos, coerência entre opções e um registro para o que é função:

In [5]:
ALIAS_GENERATORS = {"camel": to_camel, "pascal": to_pascal, "snake": to_snake}


class ModelPolicy(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True, frozen=True)

    extra: Literal["ignore", "forbid", "allow"] = "ignore"
    strict: bool = False
    frozen: bool = False
    validate_assignment: bool = False
    validate_default: bool = False
    coerce_numbers_to_str: bool = False
    str_strip_whitespace: bool = False
    str_to_lower: bool = False
    str_to_upper: bool = False
    use_enum_values: bool = False
    alias_generator: Literal["camel", "pascal", "snake"] | None = None
    validate_by_name: bool = False

    @model_validator(mode="before")
    @classmethod
    def renamed_keys(cls, data: Any) -> Any:
        if isinstance(data, dict) and "populate_by_name" in data:
            raise ValueError("populate_by_name foi renomeado: use validate_by_name")
        return data

    @model_validator(mode="after")
    def coherent(self) -> Self:
        problems = []
        if self.strict and self.coerce_numbers_to_str:
            problems.append("strict=True anula coerce_numbers_to_str")
        if self.str_to_lower and self.str_to_upper:
            problems.append("str_to_lower e str_to_upper juntos (lower vence em silêncio)")
        if self.strict and self.use_enum_values:
            problems.append("strict=True + use_enum_values: enums por valor só passam vindo de JSON")
        if problems:
            raise ValueError("; ".join(problems))
        return self

    def to_config_dict(self) -> ConfigDict:
        data = self.model_dump(exclude_defaults=True)
        if self.alias_generator:
            data["alias_generator"] = ALIAS_GENERATORS[self.alias_generator]
        return ConfigDict(**data)

O arquivo original, passando pelo `ModelPolicy`:

In [6]:
try:
    ModelPolicy.model_validate(raw_policy)
except ValidationError as ex:
    print(f"data_model.yaml: {ex.errors()[0]['msg']}")

fixed = {k: v for k, v in raw_policy.items() if k != "populate_by_name"} | {"validate_by_name": True}
try:
    ModelPolicy.model_validate(fixed)
except ValidationError as ex:
    print(f"após renomear: {ex.errors()[0]['msg']}")

for typo in [{"strct": True}, {"extra": "forbidden"}, {"strict": "yes"}]:
    try:
        ModelPolicy.model_validate(typo)
    except ValidationError as ex:
        print(f"{typo}: {ex.errors()[0]['loc']} -> {ex.errors()[0]['msg']}")

data_model.yaml: Value error, populate_by_name foi renomeado: use validate_by_name
após renomear: Value error, strict=True anula coerce_numbers_to_str; strict=True + use_enum_values: enums por valor só passam vindo de JSON
{'strct': True}: ('strct',) -> Extra inputs are not permitted
{'extra': 'forbidden'}: ('extra',) -> Input should be 'ignore', 'forbid' or 'allow'
{'strict': 'yes'}: ('strict',) -> Input should be a valid boolean


Uma versão coerente do arquivo original — decidindo, em vez de deixar o `strict` decidir em silêncio (aqui: sem `strict`, mantendo
a conversão de números e o valor dos enums):

In [7]:
corrected = """
data_config:
  validate_assignment: true
  validate_default: true
  extra: allow            # ignore (padrão) | forbid | allow
  frozen: false           # true: imutável (hashable só se todos os campos forem)
  strict: false           # true anularia coerce_numbers_to_str e exigiria membros de Enum em dicts
  coerce_numbers_to_str: true
  str_strip_whitespace: true
  str_to_lower: true
  use_enum_values: true
  validate_by_name: true  # antigo populate_by_name
"""
policy = ModelPolicy.model_validate(yaml.safe_load(corrected)["data_config"])


class Fixed(BaseModel):
    model_config = policy.to_config_dict()
    code: str
    color: Color


print(f"Fixed.model_validate({{'code': 10, 'color': 'red'}}): {Fixed.model_validate({'code': 10, 'color': 'red'})!r}")

Fixed.model_validate({'code': 10, 'color': 'red'}): Fixed(code='10', color='red')


## 4. Perfis nomeados ([`configs/policies.yaml`](configs/policies.yaml))

Em vez de uma política solta, um conjunto **fechado** de perfis com nome. Os modelos escolhem o perfil; o arquivo define o que
cada perfil significa.

In [8]:
@dataclass(frozen=True)
class DataclassPolicy:
    frozen: bool = False
    slots: bool = False
    kw_only: bool = False
    order: bool = False
    eq: bool = True

    def __post_init__(self):
        if self.order and not self.eq:
            raise ValueError("order=True exige eq=True")


class PolicyFile(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)
    model_profiles: dict[str, ModelPolicy]
    dataclass_profiles: dict[str, DataclassPolicy]


policies = PolicyFile.model_validate(yaml.safe_load((CONFIGS / "policies.yaml").read_text()))
print(f"perfis de modelo: {list(policies.model_profiles)}")
print(f"perfis de dataclass: {list(policies.dataclass_profiles)}")

perfis de modelo: ['api_input', 'internal', 'legacy_import']
perfis de dataclass: ['value_object', 'record']


Repare que `DataclassPolicy` é uma **dataclass stdlib**: o Pydantic a valida como parte do `PolicyFile` (tipos, chaves extras —
herdando o `extra="forbid"` — e o `__post_init__`).

Os perfis viram uma fábrica de classes base e de decoradores:

In [9]:
from functools import cache


@cache
def base_model(profile: str) -> type[BaseModel]:
    config = policies.model_profiles[profile].to_config_dict()
    return type(f"{profile.title().replace('_', '')}Base", (BaseModel,), {"model_config": config})


def record_decorator(profile: str):
    options = policies.dataclass_profiles[profile]
    return partial(dataclass, **{f.name: getattr(options, f.name) for f in fields(options)})


class Customer(base_model("api_input")):
    full_name: str
    tax_id: str


@record_decorator("value_object")
class Coordinates:
    lat: float
    lon: float


print(f"Customer.model_validate({{'fullName': ' Ada ', 'taxId': '123'}}): {Customer.model_validate({'fullName': ' Ada ', 'taxId': '123'})!r}")
print(f"Coordinates(lat=1.0, lon=2.0): {Coordinates(lat=1.0, lon=2.0)!r}")
print(f"base_model('api_input') is base_model('api_input'): {base_model('api_input') is base_model('api_input')}")

Customer.model_validate({'fullName': ' Ada ', 'taxId': '123'}): Customer(full_name='Ada', tax_id='123')
Coordinates(lat=1.0, lon=2.0): Coordinates(lat=1.0, lon=2.0)
base_model('api_input') is base_model('api_input'): True


`@cache` garante **uma** classe base por perfil — sem ele, cada chamada criaria uma classe nova, e `isinstance` entre modelos do
mesmo perfil deixaria de funcionar.

## 5. Pegadinhas das opções do `@dataclass`

In [10]:
@dataclass
class Mutable:
    x: int


try:
    @dataclass(frozen=True)
    class FrozenChild(Mutable):
        y: int
except TypeError as ex:
    print(f"frozen herdando de não-frozen: {ex}")

try:
    @dataclass(order=True, eq=False)
    class Broken:
        x: int
except ValueError as ex:
    print(f"order sem eq: {ex}")

frozen herdando de não-frozen: cannot inherit frozen dataclass from a non-frozen one
order sem eq: eq must be true if order is true


E uma que cruza com o notebook [02](02_factory_pattern.ipynb): `slots=True` faz o `@dataclass` **criar uma segunda classe**
(slots precisam existir na criação). Essa recriação chama o `__init_subclass__` da base de novo — **sem** os argumentos de classe.
O autorregistro com `name=...` quebra:

In [11]:
from typing import ClassVar


class Plugin:
    registry: dict[str, type] = {}

    def __init_subclass__(cls, *, name: str, **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin.registry[name] = cls


try:
    @dataclass(slots=True)
    class CsvPlugin(Plugin, name="csv"):
        delimiter: str = ","
except TypeError as ex:
    print(f"slots=True + argumento de classe: {ex}")

slots=True + argumento de classe: Plugin.__init_subclass__() missing 1 required keyword-only argument: 'name'


Tornar `name` opcional "conserta" o erro e cria um bug pior: o registro fica com a classe **original** (sem slots), e não com a
classe que o nome `CsvPlugin` passa a apontar:

In [12]:
class LoosePlugin:
    registry: dict[str, type] = {}

    def __init_subclass__(cls, *, name: str | None = None, **kwargs):
        super().__init_subclass__(**kwargs)
        if name is not None:
            LoosePlugin.registry[name] = cls


@dataclass(slots=True)
class CsvLoose(LoosePlugin, name="csv"):
    delimiter: str = ","


print(f"LoosePlugin.registry['csv'] is CsvLoose: {LoosePlugin.registry['csv'] is CsvLoose}")
print(f"a registrada tem __slots__: {'__slots__' in LoosePlugin.registry['csv'].__dict__}")

LoosePlugin.registry['csv'] is CsvLoose: False
a registrada tem __slots__: False


O que funciona: guardar o nome num `ClassVar` (vai junto no `__dict__` da classe recriada) e deixar a segunda chamada
**sobrescrever** o registro — ou registrar com um decorador aplicado **depois** do `@dataclass`:

In [13]:
class SafePlugin:
    registry: dict[str, type] = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        SafePlugin.registry[cls.plugin_name] = cls          # a 2ª chamada (classe com slots) sobrescreve


@dataclass(slots=True)
class CsvSafe(SafePlugin):
    plugin_name: ClassVar[str] = "csv"
    delimiter: str = ","


print(f"SafePlugin.registry['csv'] is CsvSafe: {SafePlugin.registry['csv'] is CsvSafe}")
print(f"tem __slots__: {'__slots__' in CsvSafe.__dict__}")

SafePlugin.registry['csv'] is CsvSafe: True
tem __slots__: True


## 6. Dataclass do Pydantic com política

`pydantic.dataclasses.dataclass` recebe as duas coisas: opções do `@dataclass` **e** o `config` do Pydantic:

In [14]:
from pydantic.dataclasses import dataclass as pydantic_dataclass


import warnings


def validated_record_naive(dataclass_profile: str, model_profile: str):
    options = policies.dataclass_profiles[dataclass_profile]
    kwargs = {f.name: getattr(options, f.name) for f in fields(options)}
    return pydantic_dataclass(config=policies.model_profiles[model_profile].to_config_dict(), **kwargs)


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    @validated_record_naive("value_object", "internal")
    class NaivePrice:
        amount_cents: int
        currency: str

print(f"aviso: {str(caught[0].message)[:150]}")

aviso: `frozen` is set via both the `dataclass` decorator and `config` for dataclass 'NaivePrice'.This is not recommended. The `frozen` specification on `dat


O perfil `internal` tem `frozen: true` no `ConfigDict` **e** o perfil `value_object` tem `frozen: true` no `@dataclass`. O Pydantic
avisa que a mesma opção vem de dois lugares. Numa dataclass, quem manda no `frozen` é o decorador; a fábrica remove a chave do
`config`:

In [15]:
def validated_record(dataclass_profile: str, model_profile: str):
    options = policies.dataclass_profiles[dataclass_profile]
    kwargs = {f.name: getattr(options, f.name) for f in fields(options)}
    config = dict(policies.model_profiles[model_profile].to_config_dict())
    config.pop("frozen", None)                           # o @dataclass decide o frozen
    return pydantic_dataclass(config=ConfigDict(**config), **kwargs)


@validated_record("value_object", "internal")
class Price:
    amount_cents: int
    currency: str


print(f"Price(amount_cents=100, currency='BRL'): {Price(amount_cents=100, currency='BRL')!r}")
try:
    Price(amount_cents="100", currency="BRL")
except ValidationError as ex:
    print(f"perfil internal (strict): {ex.errors()[0]['msg']}")
try:
    Price(amount_cents=100, currency="BRL").currency = "USD"
except Exception as ex:
    print(f"frozen (pelo decorador): {type(ex).__name__}")

Price(amount_cents=100, currency='BRL'): Price(amount_cents=100, currency='BRL')
perfil internal (strict): Input should be a valid integer
frozen (pelo decorador): FrozenInstanceError


## 7. O risco: perfil por ambiente muda o contrato

Se o perfil puder variar por ambiente, o **mesmo payload** tem destinos diferentes:

In [16]:
payload = {"fullName": "Ada", "taxId": "123", "nickname": "a"}
for profile in ["api_input", "legacy_import"]:
    Model = type(f"Customer_{profile}", (base_model(profile),), {"__annotations__": {"full_name": str, "tax_id": str}})
    try:
        print(f"{profile}: aceito -> {Model.model_validate(payload)!r}")
    except ValidationError as ex:
        print(f"{profile}: rejeitado -> {[e['type'] for e in ex.errors()]}")

api_input: rejeitado -> ['extra_forbidden']
legacy_import: rejeitado -> ['missing', 'missing']


Em `api_input` o payload é rejeitado (campo extra); em `legacy_import`, os aliases não existem e os campos obrigatórios "somem".
Se testes rodam com um perfil e produção com outro, o teste valida um contrato que não está em produção.

Regra: o **perfil de cada modelo** é decisão de código (o modelo declara `base_model("api_input")`); o arquivo pode definir **o que**
cada perfil significa, mas igual em todos os ambientes.

## 8. Resumo

| abordagem | revisão | erros de digitação | quando |
|---|---|---|---|
| presets em código | diff do código | o editor acusa | **padrão** |
| `ConfigDict(**yaml)` direto | diff do YAML | **silenciosos** | nunca |
| arquivo de perfis + `ModelPolicy` | diff do YAML | erro na carga, com caminho | muitos modelos/serviços compartilhando perfis; modelos gerados |

In [17]:
assert set(policies.model_profiles) == {"api_input", "internal", "legacy_import"}
assert SafePlugin.registry["csv"] is CsvSafe
print("verificações: ok")

verificações: ok
